# Knee MRI v4 -- DINOv2 Backbone + Optional Fine-Tuning, Decoupled Train/Inference

Builds on v3 (whose Section A preprocessing, incl. the plane-aware laterality fix, is carried
over unchanged). Three structural changes:

1. **Swappable backbone** -- `BACKBONE = 'dinov2_small' | 'efficientnet_b0'`. DINOv2-Small
   (ViT-S/14, 384-dim) is loaded through `transformers` (`timm` is not installed in
   `torch_gpu`; `transformers` 5.5.4 is).
2. **Optional fine-tuning** -- `BACKBONE_MODE = 'frozen' | 'finetune'`. Frozen keeps v3's
   fast precompute-once architecture. Finetune unfreezes the last `UNFREEZE_LAST_N` blocks
   and trains end-to-end on images.
3. **Decoupled train/inference** -- `RUN_MODE = 'train' | 'inference'`. Inference mode skips
   training entirely and loads checkpoints from `CHECKPOINT_INPUT_DIR` (a Kaggle Dataset you
   upload after training locally).

## Read this before running anything

**Unfreezing destroys the precompute speedup.** v2/v3 ran ~9s/epoch only because the backbone
was frozen and features were computed once. End-to-end training must push every slice through
the backbone every epoch, with backward passes. At v3's slice budget (12 slices x 3 series x
3 planes = **108 images/study**, ~3525 studies/fold ≈ 380K images/epoch) that is roughly
**21 min/epoch → ~3.5h/fold → ~17h for 5 folds** on an RTX 4050 -- not viable.

So the fine-tune path uses a **separate, smaller slice budget** (`FT_N_SLICES=8`,
`FT_MAX_SERIES_PER_PLANE=1` → 24 images/study). Benchmarked on an actual RTX 4050 6GB
(DINOv2-S, last 4 blocks unfrozen, AMP):

| batch | VRAM | min/epoch | 5 folds x 8 epochs |
|------:|-----:|----------:|-------------------:|
| 1  | 0.66 GB | 7.3 | 4.9 h |
| 2  | 1.12 GB | 3.2 | 2.2 h |
| 4  | 2.04 GB | 4.7 | 3.1 h |
| 8  | 3.88 GB | 4.8 | 3.2 h |
| 12 | 5.72 GB | 9.6 | 6.4 h |

**VRAM is not the binding constraint** -- even batch 8 uses under 4GB of 6.44GB. Throughput is,
and it plateaus around batch 2-8. Defaults are set to batch 4 (2GB, plenty of headroom).
These are pure-GPU numbers; with `TRAIN_NUM_WORKERS=0` (unavoidable in a Windows notebook)
disk reads serialize against compute, so expect real epochs somewhat slower. If that bites,
run the fine-tune as a plain `.py` script instead of a notebook so workers become usable.

**Do not train inside the submission notebook.** Train locally → save checkpoints → upload as
a Kaggle Dataset → run this notebook on Kaggle with `RUN_MODE='inference'`. Training then
costs nothing against Kaggle's 9-hour limit, which is what makes expensive fine-tuning
practical at all.

## On measurement (this matters more than the architecture right now)

With 58 gold rows the standard error on an AUC is roughly ±0.05-0.07, and a single-fold run
sees only ~13 of them. v3's post-fix single-fold result (`gold AUC 0.7294`, n=13, with
`Lateral OA 1.0000`) is not comparable to v2's 5-fold 0.7259 (n=58) -- perfect per-class AUCs
are what tiny samples produce by luck. Real evidence the gold metric is noisy-pessimistic:
the actual leaderboard scored v2 at **0.775** while its own gold OOF said 0.7259.

Section F now refuses to print a comparison verdict unless all 5 folds ran, and reports the
gold-n alongside every number. **Steer by the noisy AUC (n=4407)**; treat gold AUC as a coarse
sanity check.

In [1]:
import os
import gc
import time
import random
import threading
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from sklearn.metrics import roc_auc_score
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
import pydicom

TARGET_COLS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
    'Synovitis', "Baker's", 'Contusion', 'Fracture',
]
PLANES = ['Sagittal', 'Coronal', 'Axial']
PLANE_KEYS = ['sagittal', 'coronal', 'axial']
DISAGREEMENT_COLS = [f'{c}__disagreement' for c in TARGET_COLS]

REGION_GROUPS = {
    'ligament': ['ACL', 'MCL'],
    'meniscus': ['Medial Meniscus', 'Lateral Meniscus'],
    'oa':       ['Medial OA', 'Lateral OA', 'PF OA'],
    'fluid':    ['Effusion', 'Synovitis', "Baker's"],
    'trauma':   ['Contusion', 'Fracture'],
}

In [2]:
# -----------------------------------------------------------------------
# Config
# -----------------------------------------------------------------------
IS_KAGGLE = os.path.isdir('/kaggle')

# --- the three switches that define what this run does ---------------------
BACKBONE = 'dinov2_small'      # 'dinov2_small' | 'efficientnet_b0'
BACKBONE_MODE = 'frozen'       # 'frozen' (fast precompute path) | 'finetune' (end-to-end)
RUN_MODE = 'train'             # 'train' | 'inference' (load checkpoints, skip training)
# ---------------------------------------------------------------------------

DATA_DIR = 'D:/Coding/kaggle/RSNA/data'   # <-- on Kaggle, change to your dataset mount path
MAIN_DIR = f'{DATA_DIR}/main_data'

CONSENSUS_LABELS_CSV = f'{MAIN_DIR}/final/train_folds_consensus.csv'
SERIES_META_CSV = f'{MAIN_DIR}/train_series.csv'
GOLD_LABELS_CSV = f'{MAIN_DIR}/to_use/available_labels.csv'
TEST_CSV = f'{MAIN_DIR}/test.csv'
TEST_SERIES_CSV = f'{MAIN_DIR}/test_series.csv'
SAMPLE_SUB_CSV = f'{MAIN_DIR}/sample_submission.csv'

if IS_KAGGLE:
    KAGGLE_COMP_DIR = '/kaggle/input/competitions/rsna-knee-abnormality-detection'
    TRAIN_DICOM_DIR = f'{KAGGLE_COMP_DIR}/train_series'
    TEST_DICOM_DIR = f'{KAGGLE_COMP_DIR}/test_series'
else:
    TRAIN_DICOM_DIR = f'{DATA_DIR}/Raw_dicom/train_series'
    TEST_DICOM_DIR = f'{DATA_DIR}/Raw_dicom/test_series'

OUTPUT_DIR = '/kaggle/working' if IS_KAGGLE else 'D:/Coding/kaggle/RSNA/output/v4'
PREPROCESSED_DIR = f'{OUTPUT_DIR}/preprocessed'
FEATURE_CACHE_DIR = f'{OUTPUT_DIR}/feature_cache'
TEST_FEATURE_CACHE_DIR = f'{OUTPUT_DIR}/test_feature_cache'
for _d in (OUTPUT_DIR, PREPROCESSED_DIR, FEATURE_CACHE_DIR, TEST_FEATURE_CACHE_DIR):
    os.makedirs(_d, exist_ok=True)

# Reuse v3's preprocessed images if you already built them -- Section A's logic is unchanged
# from v3-post-fix, so those .npy files are still valid. Point this at v3's folder to skip
# redoing the DICOM decode entirely.
REUSE_PREPROCESSED_FROM = 'D:/Coding/kaggle/RSNA/output/v3/preprocessed' if not IS_KAGGLE else None
if REUSE_PREPROCESSED_FROM and os.path.isdir(REUSE_PREPROCESSED_FROM):
    PREPROCESSED_DIR = REUSE_PREPROCESSED_FROM
    print(f'Reusing v3 preprocessed images: {PREPROCESSED_DIR}')

# Offline weights (Kaggle commit runs have no internet).
# DINOv2: save locally once with
#   AutoModel.from_pretrained('facebook/dinov2-small').save_pretrained('dinov2_small_local')
# then upload that folder as a Kaggle Dataset and point this at it.
DINOV2_WEIGHTS_PATH = None     # e.g. '/kaggle/input/datasets/<you>/dinov2-small/dinov2_small_local'
EFFICIENTNET_WEIGHTS_PATH = None

# Checkpoints to load when RUN_MODE == 'inference' (upload your locally-trained .pt files).
CHECKPOINT_INPUT_DIR = None    # e.g. '/kaggle/input/datasets/<you>/knee-v4-checkpoints'

# Frozen-path slice budget (features precomputed once -- can afford to be generous)
N_SLICES = 12
MAX_SERIES_PER_PLANE = 3

# Fine-tune slice budget (every slice costs a fwd+bwd every epoch -- must be much smaller)
FT_N_SLICES = 8
FT_MAX_SERIES_PER_PLANE = 1

IMG_SIZE = 224
UNFREEZE_LAST_N = 4            # transformer blocks (or EffNet feature stages) to unfreeze
GRAD_CHECKPOINTING = False     # turn on if you hit OOM -- trades compute for memory

PREPROCESS_BATCH_SIZE = 8
NUM_LOAD_WORKERS = max(1, (os.cpu_count() or 2) - 1)

# Frozen-path training (tiny head on cached features)
TRAIN_BATCH_SIZE = 32
TRAIN_NUM_WORKERS = 0
EPOCHS = 20
LR = 1e-3

# Fine-tune training.
# Benchmarked on an RTX 4050 6GB (DINOv2-S, last 4 blocks unfrozen, AMP, 24 images/study):
#   batch 1 -> 0.66GB, 7.3 min/epoch      batch 4 -> 2.04GB, 4.7 min/epoch
#   batch 2 -> 1.12GB, 3.2 min/epoch      batch 8 -> 3.88GB, 4.8 min/epoch
#   batch 12 -> 5.72GB, 9.6 min/epoch (degrades -- memory pressure/throttling)
# => 5 folds x 8 epochs lands around 2-3h. VRAM is not the binding constraint; throughput is.
FT_BATCH_SIZE = 4              # studies per step; 1 study = 3 planes x FT_N_SLICES images
FT_GRAD_ACCUM = 4              # effective batch = FT_BATCH_SIZE * FT_GRAD_ACCUM = 16
FT_EPOCHS = 8
FT_HEAD_LR = 1e-3
FT_BACKBONE_LR = 1e-5          # discriminative LR -- backbone moves far slower than the head

WEIGHT_DECAY = 1e-4
GOLD_WEIGHT = 8.0
MIN_LABEL_CONFIDENCE = 0.2
USE_AMP = torch.cuda.is_available()

FOLDS_TO_RUN = list(range(5))  # anything less makes the gold-AUC comparison meaningless (see Section F)
SEED = 42


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Local'} | OUTPUT_DIR: {OUTPUT_DIR}")
print(f'Backbone: {BACKBONE} | mode: {BACKBONE_MODE} | run: {RUN_MODE}')
print(f'Device: {device} | loader threads: {NUM_LOAD_WORKERS} (of {os.cpu_count()} logical cores)')
if BACKBONE_MODE == 'finetune':
    _per_study = 3 * FT_N_SLICES * FT_MAX_SERIES_PER_PLANE
    print(f'Fine-tune budget: {_per_study} images/study (vs {3 * N_SLICES * MAX_SERIES_PER_PLANE} on the frozen path)')

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

Reusing v3 preprocessed images: D:/Coding/kaggle/RSNA/output/v3/preprocessed
Environment: Local | OUTPUT_DIR: D:/Coding/kaggle/RSNA/output/v4
Backbone: dinov2_small | mode: frozen | run: train
Device: cuda | loader threads: 11 (of 12 logical cores)


## 1. Load series metadata (train + test)

In [3]:
train_series_df = pd.read_csv(SERIES_META_CSV)
test_series_df = pd.read_csv(TEST_SERIES_CSV)
test_df = pd.read_csv(TEST_CSV)
consensus_df = pd.read_csv(CONSENSUS_LABELS_CSV)
gold_df = pd.read_csv(GOLD_LABELS_CSV)
gold_ids = set(gold_df['StudyInstanceUID'])

print(f'Train studies: {consensus_df["StudyInstanceUID"].nunique()} | train series: {len(train_series_df)}')
print(f'Test studies: {test_series_df["StudyInstanceUID"].nunique()} | test series: {len(test_series_df)}')
print(f'Gold rows: {len(gold_ids)}')


def build_all_series_by_plane(series_meta, max_per_plane):
    """(study, plane) -> up to max_per_plane SeriesInstanceUIDs, fluid-sensitive series first."""
    result = {}
    series_meta = series_meta.sort_values(['Fluid_Sensitive', 'SeriesInstanceUID'], ascending=[False, True])
    for (uid, plane), grp in series_meta.groupby(['StudyInstanceUID', 'Anatomical_Plane']):
        result[(uid, plane)] = grp['SeriesInstanceUID'].tolist()[:max_per_plane]
    return result


_max_series = MAX_SERIES_PER_PLANE if BACKBONE_MODE == 'frozen' else FT_MAX_SERIES_PER_PLANE
_n_slices = N_SLICES if BACKBONE_MODE == 'frozen' else FT_N_SLICES
train_series_by_plane = build_all_series_by_plane(train_series_df, _max_series)
test_series_by_plane = build_all_series_by_plane(test_series_df, _max_series)
print(f'Using {_n_slices} slices x up to {_max_series} series per plane ({BACKBONE_MODE} mode)')

Train studies: 4407 | train series: 24371
Test studies: 3 | test series: 15
Gold rows: 58
Using 12 slices x up to 3 series per plane (frozen mode)


## Section A: DICOM preprocessing

Unchanged from v3-post-fix -- spatially-correct slice ordering, rescale/photometric handling,
per-series percentile normalization, and **plane-aware** laterality mirroring (Coronal/Axial
only; Sagittal's horizontal axis is anterior-posterior, not left-right). If `PREPROCESSED_DIR`
was repointed at v3's output above, this whole section is a no-op.

In [4]:
def _order_series_files(series_dir):
    fnames = [f for f in os.listdir(series_dir) if f.lower().endswith('.dcm')]
    entries = []
    for fname in fnames:
        try:
            ds = pydicom.dcmread(os.path.join(series_dir, fname))
        except Exception:
            continue
        entries.append([None, fname, ds])
    if not entries:
        return []

    have_geometry = True
    for e in entries:
        try:
            iop = np.array(e[2].ImageOrientationPatient, dtype=np.float64)
            ipp = np.array(e[2].ImagePositionPatient, dtype=np.float64)
            normal = np.cross(iop[0:3], iop[3:6])
            e[0] = float(np.dot(normal, ipp))
        except Exception:
            have_geometry = False
            break
    if have_geometry and len({round(e[0], 3) for e in entries}) > 1:
        entries.sort(key=lambda e: e[0])
        return [(e[1], e[2]) for e in entries]

    try:
        nums = [int(getattr(e[2], 'InstanceNumber', -1)) for e in entries]
        if len(set(nums)) == len(entries) and all(n >= 0 for n in nums):
            order = sorted(range(len(entries)), key=lambda i: nums[i])
            return [(entries[i][1], entries[i][2]) for i in order]
    except Exception:
        pass

    entries.sort(key=lambda e: e[1])
    return [(e[1], e[2]) for e in entries]


def _decode_slice(ds):
    arr = ds.pixel_array.astype(np.float32)
    if arr.ndim == 3:
        arr = arr[arr.shape[0] // 2]
    arr = arr * float(getattr(ds, 'RescaleSlope', 1)) + float(getattr(ds, 'RescaleIntercept', 0))
    if str(getattr(ds, 'PhotometricInterpretation', '')) == 'MONOCHROME1':
        arr = arr.max() - arr
    return arr


def _tag(ds, name, cast=None, default=None):
    if not hasattr(ds, name):
        return default
    try:
        val = getattr(ds, name)
        return cast(val) if cast else val
    except Exception:
        return default


def preprocess_series(series_dir, img_size, anatomical_plane=None):
    ordered = _order_series_files(series_dir)
    if not ordered:
        return None, {}

    slices = []
    for _, ds in ordered:
        try:
            slices.append(_decode_slice(ds))
        except Exception:
            continue
    if not slices:
        return None, {}

    stack = np.stack(slices, axis=0)
    lo, hi = np.percentile(stack, [1.0, 99.0])
    hi = max(hi, lo + 1.0)
    stack = (np.clip((stack - lo) / (hi - lo), 0, 1) * 255.0).astype(np.uint8)

    first_ds = ordered[0][1]
    laterality = str(_tag(first_ds, 'Laterality', str, '') or '').strip().upper()
    mirrored = False
    # Only Coronal/Axial have an image-column axis corresponding to true patient left-right.
    if laterality == 'R' and anatomical_plane in ('Coronal', 'Axial'):
        stack = stack[:, :, ::-1]
        mirrored = True

    resized = np.stack([
        np.array(Image.fromarray(np.ascontiguousarray(s)).resize((img_size, img_size), Image.BILINEAR),
                 dtype=np.uint8)
        for s in stack
    ])

    spacing = _tag(first_ds, 'PixelSpacing')
    meta = {
        'n_dcm_files': len(ordered), 'n_slices_decoded': len(slices),
        'native_rows': _tag(first_ds, 'Rows', int), 'native_cols': _tag(first_ds, 'Columns', int),
        'pixel_spacing_row': float(spacing[0]) if spacing is not None else None,
        'pixel_spacing_col': float(spacing[1]) if spacing is not None else None,
        'slice_thickness': _tag(first_ds, 'SliceThickness', float),
        'manufacturer': _tag(first_ds, 'Manufacturer', str, ''),
        'magnetic_field_strength': _tag(first_ds, 'MagneticFieldStrength', float),
        'photometric_interpretation': _tag(first_ds, 'PhotometricInterpretation', str, ''),
        'laterality': laterality, 'mirrored': mirrored,
    }
    return resized, meta


def preprocess_all_series(series_df, dicom_root, out_dir, img_size, num_workers, skip_existing=True):
    rows = list(series_df.itertuples())
    todo = [r for r in rows
            if not (skip_existing and os.path.exists(os.path.join(out_dir, f'{r.SeriesInstanceUID}.npy')))]
    print(f'{len(todo)} / {len(rows)} series need preprocessing ({num_workers} threads) -> {dicom_root}')
    if not todo:
        return []

    def _work(row):
        series_dir = os.path.join(dicom_root, row.StudyInstanceUID, row.SeriesInstanceUID)
        if not os.path.isdir(series_dir):
            return row.SeriesInstanceUID, None, {}
        arr, meta = preprocess_series(series_dir, img_size, anatomical_plane=row.Anatomical_Plane)
        return row.SeriesInstanceUID, arr, meta

    meta_rows, t0, done, failed = [], time.time(), 0, 0
    with ThreadPoolExecutor(max_workers=num_workers) as pool:
        for series_uid, arr, meta in pool.map(_work, todo):
            if arr is not None:
                np.save(os.path.join(out_dir, f'{series_uid}.npy'), arr)
            else:
                failed += 1
            meta['SeriesInstanceUID'] = series_uid
            meta['ok'] = arr is not None
            meta_rows.append(meta)
            done += 1
            if done % 200 == 0 or done == len(todo):
                el = time.time() - t0
                print(f'  {done}/{len(todo)} | {el:.0f}s | {done / max(el, 1e-9):.2f} series/s | {failed} failed')

    print(f'Done: {done} series in {time.time() - t0:.0f}s, {failed} failed -> {out_dir}')
    return meta_rows

In [5]:
# Section A run. Train series whose study already has cached features are skipped.
if RUN_MODE == 'train':
    _cached = {f[:-4] for f in os.listdir(FEATURE_CACHE_DIR) if f.endswith('.npz')}
    _todo_df = train_series_df[~train_series_df['StudyInstanceUID'].isin(_cached)]
    print(f'{len(_todo_df)} / {len(train_series_df)} train series to consider '
          f'({len(_cached)} studies already have cached features)')
    preprocess_all_series(_todo_df, TRAIN_DICOM_DIR, PREPROCESSED_DIR, IMG_SIZE, NUM_LOAD_WORKERS)

# Test always runs fresh -- the real hidden test data is only present during a scored run.
preprocess_all_series(test_series_df, TEST_DICOM_DIR, PREPROCESSED_DIR, IMG_SIZE,
                      NUM_LOAD_WORKERS, skip_existing=False)

24371 / 24371 train series to consider (0 studies already have cached features)
0 / 24371 series need preprocessing (11 threads) -> D:/Coding/kaggle/RSNA/data/Raw_dicom/train_series
15 / 15 series need preprocessing (11 threads) -> D:/Coding/kaggle/RSNA/data/Raw_dicom/test_series
  15/15 | 7s | 2.21 series/s | 0 failed
Done: 15 series in 7s, 0 failed -> D:/Coding/kaggle/RSNA/output/v3/preprocessed


[{'n_dcm_files': 34,
  'n_slices_decoded': 34,
  'native_rows': 960,
  'native_cols': 960,
  'pixel_spacing_row': 0.1562,
  'pixel_spacing_col': 0.1562,
  'slice_thickness': 3.0,
  'manufacturer': 'Siemens Healthineers',
  'magnetic_field_strength': 1.5,
  'photometric_interpretation': 'MONOCHROME2',
  'laterality': 'L',
  'mirrored': False,
  'SeriesInstanceUID': '1.2.826.0.1.3680043.8.498.11580656442259111255675562605155903947',
  'ok': True},
 {'n_dcm_files': 30,
  'n_slices_decoded': 30,
  'native_rows': 800,
  'native_cols': 800,
  'pixel_spacing_row': 0.1875,
  'pixel_spacing_col': 0.1875,
  'slice_thickness': 3.0,
  'manufacturer': 'Siemens Healthineers',
  'magnetic_field_strength': 1.5,
  'photometric_interpretation': 'MONOCHROME2',
  'laterality': 'L',
  'mirrored': False,
  'SeriesInstanceUID': '1.2.826.0.1.3680043.8.498.17811502614030631664517622518906646132',
  'ok': True},
 {'n_dcm_files': 30,
  'n_slices_decoded': 30,
  'native_rows': 896,
  'native_cols': 896,
  'pixel_

## Section B: backbone, pooling, heads

`SliceBackbone` gives DINOv2-Small and EfficientNet-B0 one interface so they are A/B
swappable. `mode='finetune'` unfreezes the last `UNFREEZE_LAST_N` blocks only -- full
unfreezing on 58 gold labels is a good way to overfit spectacularly.

In [6]:
class SliceBackbone(nn.Module):
    """Uniform (N,3,H,W) -> (N, out_dim) interface over DINOv2-Small and EfficientNet-B0."""

    def __init__(self, name=BACKBONE, mode=BACKBONE_MODE, unfreeze_last_n=UNFREEZE_LAST_N,
                 grad_checkpointing=GRAD_CHECKPOINTING):
        super().__init__()
        self.name = name
        if name == 'dinov2_small':
            from transformers import AutoModel
            src = DINOV2_WEIGHTS_PATH if (DINOV2_WEIGHTS_PATH and os.path.isdir(DINOV2_WEIGHTS_PATH)) \
                else 'facebook/dinov2-small'
            self.net = AutoModel.from_pretrained(src)     # needs internet unless src is a local dir
            self.out_dim = int(self.net.config.hidden_size)
            if grad_checkpointing:
                self.net.gradient_checkpointing_enable()
        elif name == 'efficientnet_b0':
            if EFFICIENTNET_WEIGHTS_PATH and os.path.exists(EFFICIENTNET_WEIGHTS_PATH):
                m = efficientnet_b0(weights=None)
                m.load_state_dict(torch.load(EFFICIENTNET_WEIGHTS_PATH, map_location='cpu'))
            else:
                m = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
            self.features, self.pool = m.features, m.avgpool
            self.out_dim = 1280
        else:
            raise ValueError(f'unknown backbone: {name}')
        self._set_trainable(mode, unfreeze_last_n)

    def _set_trainable(self, mode, unfreeze_last_n):
        for p in self.parameters():
            p.requires_grad = False
        if mode != 'finetune':
            return
        if self.name == 'dinov2_small':
            for blk in self.net.encoder.layer[-unfreeze_last_n:]:
                for p in blk.parameters():
                    p.requires_grad = True
            for p in self.net.layernorm.parameters():
                p.requires_grad = True
        else:
            stages = list(self.features.children())
            for stage in stages[-unfreeze_last_n:]:
                for p in stage.parameters():
                    p.requires_grad = True
        n_train = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f'  backbone trainable params: {n_train:,}')

    def forward(self, x):
        if self.name == 'dinov2_small':
            return self.net(pixel_values=x).last_hidden_state[:, 0]   # CLS token
        return self.pool(self.features(x)).flatten(1)


class AttentionPool(nn.Module):
    """Mask-aware attention pooling -- padded slots never contribute."""

    def __init__(self, dim, hidden=128):
        super().__init__()
        self.attn = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))

    def forward(self, x, mask=None):
        scores = self.attn(x)
        if mask is not None:
            invalid = (mask == 0).unsqueeze(-1)
            all_invalid = invalid.all(dim=1, keepdim=True)
            scores = scores.masked_fill(invalid & ~all_invalid, float('-inf'))
        return (x * torch.softmax(scores, dim=1)).sum(dim=1)


class PlaneAttentionHead(nn.Module):
    def __init__(self, dim, num_targets, hidden=128):
        super().__init__()
        self.plane_attn = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))
        self.classifier = nn.Sequential(
            nn.Linear(dim, hidden), nn.ReLU(inplace=True), nn.Dropout(0.3), nn.Linear(hidden, num_targets)
        )

    def forward(self, plane_embeds):
        w = torch.softmax(self.plane_attn(plane_embeds), dim=1)
        return self.classifier((plane_embeds * w).sum(dim=1))


class RegionGuidedHead(nn.Module):
    def __init__(self, dim, region_groups=REGION_GROUPS, target_cols=TARGET_COLS):
        super().__init__()
        self.target_cols = target_cols
        self.heads = nn.ModuleDict()
        self.target_indices = {}
        for name, targets in region_groups.items():
            self.heads[name] = PlaneAttentionHead(dim, len(targets))
            self.target_indices[name] = [target_cols.index(t) for t in targets]

    def forward(self, plane_embeds):
        B = plane_embeds.size(0)
        logits = torch.zeros(B, len(self.target_cols), device=plane_embeds.device, dtype=plane_embeds.dtype)
        for name, head in self.heads.items():
            out = head(plane_embeds)
            for i, idx in enumerate(self.target_indices[name]):
                logits[:, idx] = out[:, i]
        return logits

## Section C: data loading

Two dataset classes for the two paths:
- `KneeFeatureDataset` (frozen) -- reads cached backbone features, trains in seconds/epoch.
- `KneeImageDataset` (finetune) -- reads preprocessed images so gradients can reach the
  backbone. Augmentation is intensity jitter + random slice offset; **no horizontal flip**,
  which would undo the laterality normalization done in Section A.

In [7]:
_fallback_lock = threading.Lock()
_fallback_count = [0]


def _load_plane_images(uid, plane, series_by_plane, n_slices, max_series, jitter=False):
    """-> ((max_series*n_slices, H, W) float32 in [0,1], (max_series*n_slices,) mask)."""
    arrs, masks = [], []
    for series_uid in series_by_plane.get((uid, plane), [])[:max_series]:
        path = os.path.join(PREPROCESSED_DIR, f'{series_uid}.npy')
        if not os.path.exists(path):
            with _fallback_lock:
                _fallback_count[0] += 1
            continue
        arr = np.load(path).astype(np.float32) / 255.0
        T = arr.shape[0]
        if T >= n_slices:
            if jitter and T > n_slices:
                lo = np.random.randint(0, max(1, T - n_slices + 1))
                idx = np.linspace(lo, min(T - 1, lo + n_slices - 1), n_slices).astype(int)
            else:
                idx = np.linspace(0, T - 1, n_slices).astype(int)
            arr, m = arr[idx], np.ones(n_slices, dtype=np.float32)
        else:
            pad = np.zeros((n_slices - T, IMG_SIZE, IMG_SIZE), dtype=np.float32)
            m = np.concatenate([np.ones(T, dtype=np.float32), np.zeros(n_slices - T, dtype=np.float32)])
            arr = np.concatenate([arr, pad], axis=0)
        if jitter:
            arr = np.clip(arr * np.random.uniform(0.9, 1.1) + np.random.uniform(-0.05, 0.05), 0, 1)
        arrs.append(arr)
        masks.append(m)

    while len(arrs) < max_series:
        arrs.append(np.zeros((n_slices, IMG_SIZE, IMG_SIZE), dtype=np.float32))
        masks.append(np.zeros(n_slices, dtype=np.float32))
    return np.concatenate(arrs, axis=0), np.concatenate(masks, axis=0)


class _LabelMixin:
    def _init_labels(self, labels_df, gold_ids, gold_weight):
        self.gold_ids = gold_ids or set()
        self.gold_weight = gold_weight
        self.label_lookup = None
        self.label_weight_lookup = None
        if labels_df is None:
            return
        idx = labels_df.set_index('StudyInstanceUID')[TARGET_COLS]
        self.label_lookup = {u: idx.loc[u].astype(float).values for u in self.study_ids}
        if all(c in labels_df.columns for c in DISAGREEMENT_COLS):
            dis = labels_df.set_index('StudyInstanceUID')[DISAGREEMENT_COLS].clip(lower=0.0, upper=1.0)
            conf = (1.0 - dis).clip(lower=MIN_LABEL_CONFIDENCE, upper=1.0)
            self.label_weight_lookup = {u: conf.loc[u].astype(float).values for u in self.study_ids}

    def _add_labels(self, item, uid):
        if self.label_lookup is None:
            return item
        item['labels'] = torch.tensor(self.label_lookup[uid], dtype=torch.float32)
        item['weight'] = torch.tensor(self.gold_weight if uid in self.gold_ids else 1.0, dtype=torch.float32)
        item['label_weight'] = torch.tensor(
            self.label_weight_lookup[uid] if self.label_weight_lookup else np.ones(len(TARGET_COLS)),
            dtype=torch.float32)
        return item


class KneeFeatureDataset(Dataset, _LabelMixin):
    """Frozen path: cached backbone features + masks."""

    def __init__(self, study_ids, cache_dir, labels_df=None, gold_ids=None, gold_weight=1.0):
        self.study_ids = list(study_ids)
        self.cache_dir = cache_dir
        self._init_labels(labels_df, gold_ids, gold_weight)

    def __len__(self):
        return len(self.study_ids)

    def __getitem__(self, i):
        uid = self.study_ids[i]
        d = np.load(os.path.join(self.cache_dir, f'{uid}.npz'))
        item = {'study_uid': uid}
        for key in PLANE_KEYS:
            item[key] = torch.from_numpy(d[key]).float()
            item[f'{key}_mask'] = torch.from_numpy(d[f'{key}_mask']).float()
        return self._add_labels(item, uid)


class KneeImageDataset(Dataset, _LabelMixin):
    """Fine-tune path: preprocessed images, so gradients reach the backbone."""

    def __init__(self, study_ids, series_by_plane, labels_df=None, gold_ids=None,
                 gold_weight=1.0, is_train=False):
        self.study_ids = list(study_ids)
        self.series_by_plane = series_by_plane
        self.is_train = is_train
        self._init_labels(labels_df, gold_ids, gold_weight)

    def __len__(self):
        return len(self.study_ids)

    def __getitem__(self, i):
        uid = self.study_ids[i]
        item = {'study_uid': uid}
        for plane, key in zip(PLANES, PLANE_KEYS):
            arr, mask = _load_plane_images(uid, plane, self.series_by_plane,
                                            FT_N_SLICES, FT_MAX_SERIES_PER_PLANE, jitter=self.is_train)
            t = torch.from_numpy(arr).unsqueeze(1).repeat(1, 3, 1, 1)   # (T,3,H,W)
            item[key] = (t - IMAGENET_MEAN) / IMAGENET_STD
            item[f'{key}_mask'] = torch.from_numpy(mask)
        return self._add_labels(item, uid)


class KneeFeatureModel(nn.Module):
    """Frozen path: pooling + region head over precomputed features."""

    def __init__(self, feat_dim):
        super().__init__()
        self.slice_pool = AttentionPool(feat_dim)
        self.region_head = RegionGuidedHead(feat_dim)

    def forward(self, sag, cor, ax, sm=None, cm=None, am=None):
        embeds = torch.stack([self.slice_pool(sag, sm),
                              self.slice_pool(cor, cm),
                              self.slice_pool(ax, am)], dim=1)
        return self.region_head(embeds)


class KneeEndToEndModel(nn.Module):
    """Fine-tune path: backbone + pooling + region head, trained together."""

    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.slice_pool = AttentionPool(backbone.out_dim)
        self.region_head = RegionGuidedHead(backbone.out_dim)

    def _encode(self, x, mask):
        B, T, C, H, W = x.shape
        f = self.backbone(x.view(B * T, C, H, W)).view(B, T, -1)
        return self.slice_pool(f, mask)

    def forward(self, sag, cor, ax, sm=None, cm=None, am=None):
        embeds = torch.stack([self._encode(sag, sm),
                              self._encode(cor, cm),
                              self._encode(ax, am)], dim=1)
        return self.region_head(embeds)

## Section D: feature precompute (frozen path only)

In [8]:
def _load_study_planes(uid, series_by_plane):
    out = {}
    for plane, key in zip(PLANES, PLANE_KEYS):
        arr, mask = _load_plane_images(uid, plane, series_by_plane, N_SLICES, MAX_SERIES_PER_PLANE)
        out[key], out[f'{key}_mask'] = arr, mask
    return uid, out


@torch.no_grad()
def precompute_features(study_ids, series_by_plane, backbone, cache_dir,
                         batch_size=PREPROCESS_BATCH_SIZE, num_load_workers=NUM_LOAD_WORKERS,
                         skip_existing=True):
    todo = [u for u in study_ids
            if not (skip_existing and os.path.exists(os.path.join(cache_dir, f'{u}.npz')))]
    print(f'{len(todo)} / {len(study_ids)} studies need feature extraction')
    if not todo:
        return

    backbone.eval()
    t0, done = time.time(), 0
    _fallback_count[0] = 0
    with ThreadPoolExecutor(max_workers=num_load_workers) as pool:
        for start in range(0, len(todo), batch_size):
            batch_ids = todo[start: start + batch_size]
            results = list(pool.map(lambda u: _load_study_planes(u, series_by_plane), batch_ids))

            feats = {}
            for key in PLANE_KEYS:
                raw = np.stack([r[1][key] for r in results])
                x = torch.from_numpy(raw).unsqueeze(2).repeat(1, 1, 3, 1, 1)
                x = ((x - IMAGENET_MEAN) / IMAGENET_STD).to(device)
                B, T, C, H, W = x.shape
                with torch.autocast(device_type=device.type, enabled=USE_AMP):
                    f = backbone(x.view(B * T, C, H, W))
                feats[key] = f.float().view(B, T, -1).cpu().numpy()

            for b, (uid, planes) in enumerate(results):
                np.savez(os.path.join(cache_dir, f'{uid}.npz'),
                         **{k: feats[k][b] for k in PLANE_KEYS},
                         **{f'{k}_mask': planes[f'{k}_mask'] for k in PLANE_KEYS})

            done += len(batch_ids)
            if (start // batch_size) % 20 == 0 or done == len(todo):
                el = time.time() - t0
                print(f'  {done}/{len(todo)} studies | {el:.0f}s | {done / max(el, 1e-9):.2f} studies/s')

    print(f'Precompute finished: {done} studies in {time.time() - t0:.0f}s -> {cache_dir}')
    if _fallback_count[0] > 0:
        print(f'Note: {_fallback_count[0]} series-slots missing from PREPROCESSED_DIR (masked out).')

In [9]:
backbone = SliceBackbone().to(device)
FEAT_DIM = backbone.out_dim
print(f'{BACKBONE} ready | feature dim: {FEAT_DIM}')

if BACKBONE_MODE == 'frozen':
    if RUN_MODE == 'train':
        precompute_features(consensus_df['StudyInstanceUID'].tolist(), train_series_by_plane,
                            backbone, FEATURE_CACHE_DIR)
    precompute_features(test_df['StudyInstanceUID'].tolist(), test_series_by_plane,
                        backbone, TEST_FEATURE_CACHE_DIR, skip_existing=False)
else:
    print('finetune mode -- skipping precompute (the backbone trains end-to-end on images)')

c:\Users\nishk\anaconda3\envs\torch_gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 223/223 [00:00<00:00, 10619.82it/s]


dinov2_small ready | feature dim: 384
4407 / 4407 studies need feature extraction
  8/4407 studies | 2s | 4.62 studies/s
  168/4407 studies | 32s | 5.18 studies/s
  328/4407 studies | 63s | 5.22 studies/s
  488/4407 studies | 94s | 5.21 studies/s
  648/4407 studies | 124s | 5.21 studies/s
  808/4407 studies | 156s | 5.19 studies/s
  968/4407 studies | 187s | 5.18 studies/s
  1128/4407 studies | 217s | 5.19 studies/s
  1288/4407 studies | 248s | 5.19 studies/s
  1448/4407 studies | 279s | 5.18 studies/s
  1608/4407 studies | 310s | 5.18 studies/s
  1768/4407 studies | 341s | 5.18 studies/s
  1928/4407 studies | 373s | 5.18 studies/s
  2088/4407 studies | 403s | 5.18 studies/s
  2248/4407 studies | 434s | 5.18 studies/s
  2408/4407 studies | 465s | 5.18 studies/s
  2568/4407 studies | 496s | 5.17 studies/s
  2728/4407 studies | 528s | 5.17 studies/s
  2888/4407 studies | 559s | 5.16 studies/s
  3048/4407 studies | 590s | 5.16 studies/s
  3208/4407 studies | 621s | 5.17 studies/s
  3368/4

## Section E: loss & metrics

In [10]:
def weighted_bce(logits, targets, row_weight, label_weight, pos_weight):
    per_elem = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pos_weight, reduction='none')
    per_sample = (per_elem * label_weight).mean(dim=1)
    return (per_sample * row_weight).sum() / row_weight.sum()


def macro_auc(y_true_bin, y_prob, cols):
    per_class = {}
    for i, c in enumerate(cols):
        yt = y_true_bin[:, i]
        if len(np.unique(yt)) < 2:
            continue
        per_class[c] = roc_auc_score(yt, y_prob[:, i])
    macro = float(np.mean(list(per_class.values()))) if per_class else float('nan')
    return macro, per_class


def _to_device(batch):
    return {k: (v.to(device, non_blocking=True) if torch.is_tensor(v) else v) for k, v in batch.items()}


def _fwd(model, b):
    return model(b['sagittal'], b['coronal'], b['axial'],
                 b['sagittal_mask'], b['coronal_mask'], b['axial_mask'])


def train_one_epoch(model, loader, optimizer, pos_weight, scaler, grad_accum=1):
    model.train()
    total, seen = 0.0, 0
    optimizer.zero_grad(set_to_none=True)
    for step, batch in enumerate(loader):
        b = _to_device(batch)
        with torch.autocast(device_type=device.type, enabled=USE_AMP):
            loss = weighted_bce(_fwd(model, b), b['labels'], b['weight'], b['label_weight'], pos_weight)
        scaled = loss / grad_accum
        if USE_AMP:
            scaler.scale(scaled).backward()
        else:
            scaled.backward()
        if (step + 1) % grad_accum == 0:
            if USE_AMP:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)
        total += loss.item() * b['sagittal'].size(0)
        seen += b['sagittal'].size(0)
    return total / max(seen, 1)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    probs, uids = [], []
    for batch in loader:
        b = _to_device(batch)
        with torch.autocast(device_type=device.type, enabled=USE_AMP):
            logits = _fwd(model, b)
        probs.append(torch.sigmoid(logits.float()).cpu().numpy())
        uids.extend(batch['study_uid'])
    return np.concatenate(probs, axis=0), uids

## Section F: training (per fold)

Set `RUN_MODE='inference'` to skip this entirely and load checkpoints instead -- that is what
makes fine-tuning practical, since Kaggle's 9-hour limit then never sees the training cost.

In [11]:
def make_loaders(train_ids, val_ids):
    if BACKBONE_MODE == 'frozen':
        tr = KneeFeatureDataset(train_ids, FEATURE_CACHE_DIR, consensus_df, gold_ids, GOLD_WEIGHT)
        va = KneeFeatureDataset(val_ids, FEATURE_CACHE_DIR, consensus_df, gold_ids, GOLD_WEIGHT)
        bs, accum = TRAIN_BATCH_SIZE, 1
    else:
        tr = KneeImageDataset(train_ids, train_series_by_plane, consensus_df, gold_ids,
                              GOLD_WEIGHT, is_train=True)
        va = KneeImageDataset(val_ids, train_series_by_plane, consensus_df, gold_ids, GOLD_WEIGHT)
        bs, accum = FT_BATCH_SIZE, FT_GRAD_ACCUM
    return (DataLoader(tr, batch_size=bs, shuffle=True, num_workers=TRAIN_NUM_WORKERS,
                       pin_memory=True, drop_last=True),
            DataLoader(va, batch_size=bs, shuffle=False, num_workers=TRAIN_NUM_WORKERS, pin_memory=True),
            accum)


def build_model():
    if BACKBONE_MODE == 'frozen':
        return KneeFeatureModel(FEAT_DIM).to(device)
    return KneeEndToEndModel(SliceBackbone()).to(device)


def build_optimizer(model):
    if BACKBONE_MODE == 'frozen':
        return torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    bb = [p for n, p in model.named_parameters() if n.startswith('backbone.') and p.requires_grad]
    head = [p for n, p in model.named_parameters() if not n.startswith('backbone.') and p.requires_grad]
    return torch.optim.AdamW([{'params': bb, 'lr': FT_BACKBONE_LR},
                              {'params': head, 'lr': FT_HEAD_LR}], weight_decay=WEIGHT_DECAY)


consensus_indexed = consensus_df.set_index('StudyInstanceUID')
gold_indexed = gold_df.set_index('StudyInstanceUID')
oof_probs, fold_ckpts = {}, {}
n_epochs = EPOCHS if BACKBONE_MODE == 'frozen' else FT_EPOCHS

if RUN_MODE == 'train':
    for fold in FOLDS_TO_RUN:
        print(f"\n{'=' * 72}\nFold {fold}  ({BACKBONE} / {BACKBONE_MODE})\n{'=' * 72}")
        train_ids = consensus_df.loc[consensus_df.fold != fold, 'StudyInstanceUID'].tolist()
        val_ids = consensus_df.loc[consensus_df.fold == fold, 'StudyInstanceUID'].tolist()
        train_loader, val_loader, grad_accum = make_loaders(train_ids, val_ids)
        print(f'train {len(train_ids)} | val {len(val_ids)} | grad_accum {grad_accum}')

        prevalence = consensus_df.loc[consensus_df.fold != fold, TARGET_COLS].ge(0.5).mean().values
        pos_weight = torch.tensor(np.clip((1 - prevalence) / np.clip(prevalence, 1e-3, None), 1.0, 10.0),
                                   dtype=torch.float32).to(device)

        model = build_model()
        optimizer = build_optimizer(model)
        scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
        best_gold, ckpt_path = -1.0, os.path.join(OUTPUT_DIR, f'v4_{BACKBONE}_{BACKBONE_MODE}_fold{fold}.pt')

        for epoch in range(1, n_epochs + 1):
            t0 = time.time()
            loss = train_one_epoch(model, train_loader, optimizer, pos_weight, scaler, grad_accum)
            probs, uids = predict(model, val_loader)

            soft = consensus_indexed.loc[uids, TARGET_COLS].values
            noisy, _ = macro_auc((soft >= 0.5).astype(int), probs, TARGET_COLS)
            gmask = np.array([u in gold_ids for u in uids])
            if gmask.any():
                g_uids = [u for u, m in zip(uids, gmask) if m]
                gold, _ = macro_auc(gold_indexed.loc[g_uids, TARGET_COLS].values.astype(int),
                                    probs[gmask], TARGET_COLS)
            else:
                gold = float('nan')

            mark = ''
            if not np.isnan(gold) and gold > best_gold:
                best_gold, mark = gold, ' *'
                torch.save(model.state_dict(), ckpt_path)
            print(f'epoch {epoch:>2}/{n_epochs} | loss {loss:.4f} | noisy AUC {noisy:.4f} | '
                  f'gold AUC {gold:.4f} (n={int(gmask.sum())}) | {time.time() - t0:>6.1f}s{mark}')

        if not os.path.exists(ckpt_path):
            torch.save(model.state_dict(), ckpt_path)
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        probs, uids = predict(model, val_loader)
        for u, p in zip(uids, probs):
            oof_probs[u] = p
        fold_ckpts[fold] = ckpt_path
        print(f'Fold {fold} best gold AUC: {best_gold:.4f} -> {ckpt_path}')

        del model, optimizer
        gc.collect()
        torch.cuda.empty_cache()
else:
    src = CHECKPOINT_INPUT_DIR or OUTPUT_DIR
    for fold in FOLDS_TO_RUN:
        p = os.path.join(src, f'v4_{BACKBONE}_{BACKBONE_MODE}_fold{fold}.pt')
        if os.path.exists(p):
            fold_ckpts[fold] = p
    print(f'inference mode -- loaded {len(fold_ckpts)} checkpoint(s) from {src}')
    if not fold_ckpts:
        raise FileNotFoundError(f'No checkpoints found in {src}. Set CHECKPOINT_INPUT_DIR.')


Fold 0  (dinov2_small / frozen)
train 3525 | val 882 | grad_accum 1
epoch  1/20 | loss 0.8055 | noisy AUC 0.7082 | gold AUC 0.6537 (n=13) |   44.9s *
epoch  2/20 | loss 0.7679 | noisy AUC 0.7335 | gold AUC 0.6855 (n=13) |    6.6s *
epoch  3/20 | loss 0.7545 | noisy AUC 0.7444 | gold AUC 0.7195 (n=13) |    6.2s *
epoch  4/20 | loss 0.7425 | noisy AUC 0.7560 | gold AUC 0.7256 (n=13) |    6.3s *
epoch  5/20 | loss 0.7326 | noisy AUC 0.7543 | gold AUC 0.7300 (n=13) |    6.2s *
epoch  6/20 | loss 0.7272 | noisy AUC 0.7569 | gold AUC 0.7025 (n=13) |    6.2s
epoch  7/20 | loss 0.7192 | noisy AUC 0.7546 | gold AUC 0.7369 (n=13) |    6.2s *
epoch  8/20 | loss 0.7093 | noisy AUC 0.7621 | gold AUC 0.7237 (n=13) |    5.9s
epoch  9/20 | loss 0.7066 | noisy AUC 0.7601 | gold AUC 0.7306 (n=13) |    6.2s
epoch 10/20 | loss 0.6982 | noisy AUC 0.7692 | gold AUC 0.7130 (n=13) |    5.8s
epoch 11/20 | loss 0.6947 | noisy AUC 0.7697 | gold AUC 0.7290 (n=13) |    6.1s
epoch 12/20 | loss 0.6897 | noisy AUC 0

## Section G: out-of-fold summary

In [12]:
if RUN_MODE == 'train' and oof_probs:
    uids = list(oof_probs.keys())
    mat = np.stack([oof_probs[u] for u in uids])
    soft = consensus_indexed.loc[uids, TARGET_COLS].values
    noisy_macro, _ = macro_auc((soft >= 0.5).astype(int), mat, TARGET_COLS)

    gmask = np.array([u in gold_ids for u in uids])
    g_uids = [u for u, m in zip(uids, gmask) if m]
    gold_macro, gold_per_class = macro_auc(
        gold_indexed.loc[g_uids, TARGET_COLS].values.astype(int), mat[gmask], TARGET_COLS)

    print(f'OOF studies: {len(uids)} | folds run: {sorted(FOLDS_TO_RUN)} | gold rows: {len(g_uids)}')
    print(f'Noisy macro AUC (n={len(uids)}):  {noisy_macro:.4f}   <-- steer by this one')
    print(f'Gold  macro AUC (n={len(g_uids)}): {gold_macro:.4f}')

    if len(FOLDS_TO_RUN) < 5:
        print(f'\n!! Only {len(FOLDS_TO_RUN)} fold(s) ran, so gold AUC here uses ~{len(g_uids)} rows.')
        print('!! That is NOT comparable to any 5-fold number (v2: 0.7259 @ n=58). Run all 5 folds first.')
    else:
        print(f'\nReference: v2 = 0.7259 gold / 0.7372 noisy (5-fold), real LB 0.775.')
        print(f'At n={len(g_uids)} the gold SE is roughly +/-0.06 -- differences under ~0.1 are not resolvable.')

    print('\nPer-class gold AUC:')
    for c, v in sorted(gold_per_class.items(), key=lambda x: -x[1]):
        print(f'  {c:<20} {v:.4f}')

OOF studies: 4407 | folds run: [0, 1, 2, 3, 4] | gold rows: 58
Noisy macro AUC (n=4407):  0.7493   <-- steer by this one
Gold  macro AUC (n=58): 0.7552

Reference: v2 = 0.7259 gold / 0.7372 noisy (5-fold), real LB 0.775.
At n=58 the gold SE is roughly +/-0.06 -- differences under ~0.1 are not resolvable.

Per-class gold AUC:
  Medial OA            0.8915
  Lateral OA           0.8511
  Baker's              0.8043
  Fracture             0.7931
  Contusion            0.7773
  Lateral Meniscus     0.7714
  Effusion             0.7466
  PF OA                0.7387
  MCL                  0.7324
  Medial Meniscus      0.6911
  Synovitis            0.6619
  ACL                  0.6029


## Section H: inference + submission

In [13]:
test_ids = test_df['StudyInstanceUID'].tolist()

if BACKBONE_MODE == 'frozen':
    test_ds = KneeFeatureDataset(test_ids, TEST_FEATURE_CACHE_DIR)
    bs = TRAIN_BATCH_SIZE
else:
    test_ds = KneeImageDataset(test_ids, test_series_by_plane)
    bs = FT_BATCH_SIZE
test_loader = DataLoader(test_ds, batch_size=bs, shuffle=False, num_workers=TRAIN_NUM_WORKERS)

fold_preds, test_uids = [], None
for fold, ckpt_path in sorted(fold_ckpts.items()):
    model = build_model()
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    probs, uids = predict(model, test_loader)
    fold_preds.append(probs)
    test_uids = uids
    del model
    gc.collect()
    torch.cuda.empty_cache()

sub = pd.DataFrame(np.mean(fold_preds, axis=0), columns=TARGET_COLS)
sub.insert(0, 'StudyInstanceUID', test_uids)
sub = sub[pd.read_csv(SAMPLE_SUB_CSV).columns]
sub_path = os.path.join('/kaggle/working' if IS_KAGGLE else OUTPUT_DIR, 'submission.csv')
sub.to_csv(sub_path, index=False)
print(f'Saved: {sub_path}  ({len(sub)} rows, ensembled over {len(fold_preds)} fold(s))')
sub.head()

Saved: D:/Coding/kaggle/RSNA/output/v4\submission.csv  (3 rows, ensembled over 5 fold(s))


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.318782,0.441669,0.476886,0.447284,0.742347,0.726358,0.647039,0.532525,0.551308,0.616993,0.328694,0.366401
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.477866,0.491801,0.428299,0.409902,0.581865,0.607572,0.562037,0.639613,0.846251,0.569377,0.484833,0.493550
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.238120,0.457588,0.544884,0.514952,0.601775,0.604118,0.571140,0.442337,0.470409,0.467928,0.345802,0.390388


## Suggested run order

1. **Cheap A/B first** (`BACKBONE_MODE='frozen'`, all 5 folds) -- run once with
   `BACKBONE='efficientnet_b0'` and once with `'dinov2_small'`. Section D is the only extra
   cost and features cache, so this is hours, not days. If DINOv2's frozen features don't beat
   EfficientNet's, fine-tuning DINOv2 is a much longer shot.
2. **Then fine-tune the winner** (`BACKBONE_MODE='finetune'`) locally overnight. Watch the
   first epoch's wall-clock before committing to 5 folds.
3. **Upload `v4_*_fold*.pt` as a Kaggle Dataset**, set `CHECKPOINT_INPUT_DIR` and
   `RUN_MODE='inference'`, and submit. Training never touches Kaggle's clock.

For offline DINOv2 weights on Kaggle, run this locally once and upload the folder:
```python
from transformers import AutoModel
AutoModel.from_pretrained('facebook/dinov2-small').save_pretrained('dinov2_small_local')
```